In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/Food_Delivery_Time_Prediction.csv')

df.head()

In [ ]:
# 1. Ver o tamanho do dataset 
print("Dimensões do dataset:", df.shape)

# 2. Verificar os tipos de dados e valores nulos
print("\n--- Informações Gerais ---")
df.info()

# 3. Verificar se há linhas duplicadas
print("\nValores duplicados:", df.duplicated().sum())

# 4. Estatísticas descritivas das colunas 
print("\n--- Estatísticas ---")
df.describe().T 

In [ ]:
# Configurando o tamanho dos gráficos
plt.figure(figsize=(14, 6))

# 1. Gráfico da distribuição do tempo de entrega
plt.subplot(1, 2, 1)
sns.histplot(df['Time_taken_min'], bins=50, kde=True, color='blue')
plt.title('Distribuição do Tempo de Entrega')
plt.xlabel('Minutos')
plt.ylabel('Frequência')

# 2. Boxplot para ver se há valores muito fora do padrão
plt.subplot(1, 2, 2)
sns.boxplot(y=df['Time_taken_min'], color='orange')
plt.title('Boxplot do Tempo de Entrega')
plt.ylabel('Minutos')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

# 1. Tempo de entrega baseado no Trânsito
plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='Traffic_Level', y='Time_taken_min', palette='Set2')
plt.title('Impacto do Trânsito no Tempo de Entrega')
plt.xlabel('Nível de Trânsito')
plt.ylabel('Minutos')

# 2. Tempo de entrega por Clima
plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='Weather', y='Time_taken_min', palette='Set3')
plt.title('Impacto do Clima no Tempo de Entrega')
plt.xlabel('Clima')
plt.ylabel('Minutos')

plt.tight_layout()
plt.show()

In [ ]:
# Analisar a distância e o tempo
plt.figure(figsize=(10, 6))

sns.scatterplot(data=df, x='Road_Distance_km', y='Time_taken_min', alpha=0.3, color='purple')
plt.title('Relação entre Distância Rodoviária e Tempo de Entrega')
plt.xlabel('Distância (km)')
plt.ylabel('Tempo de Entrega (Minutos)')
plt.show()

# Vamos ver a matriz de correlação apenas das variáveis numéricas
colunas_numericas = ['Preparation_Time_Min', 'Road_Distance_km', 'Number_of_Signals', 'Average_Speed_kmph', 'Time_taken_min']
plt.figure(figsize=(8, 5))
sns.heatmap(df[colunas_numericas].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação (Variáveis Numéricas)')
plt.show()

In [ ]:
# 1. Separar o alvo (y) das features (X)
y = df['Time_taken_min']

# 2. Criar o X, removendo o alvo e as colunas que causam vazamento de dados
colunas_para_remover = ['Time_taken_min', 'Order_ID', 'Order_Date', 'Average_Speed_kmph', 'Delivery_Distance_Category']
X = df.drop(columns=colunas_para_remover)

# 3. Codificar variáveis
# O pd.get_dummies faz o One-Hot Encoding
X_encoded = pd.get_dummies(X, drop_first=True)

# Ver como ficou o X final
print("Formato original do X:", X.shape)
print("Formato após One-Hot Encoding:", X_encoded.shape)
X_encoded.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Separando os dados: 80% para treino, 20% para teste.
# random_state=42 garante a mesma divisão
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print(f"Tamanho do treino: {X_train.shape[0]} pedidos")
print(f"Tamanho do teste: {X_test.shape[0]} pedidos")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 1. Criar o modelo de Regressão Linear
lr_model = LinearRegression()

# 2. Treinar o modelo com os dados de treino
lr_model.fit(X_train, y_train)

# 3. Fazer previsões nos dados de teste
y_pred_lr = lr_model.predict(X_test)

# 4. Calcular as métricas
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = root_mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("--- Regressão Linear ---")
print(f"MAE: {mae_lr:.2f} minutos")
print(f"RMSE: {rmse_lr:.2f} minutos")
print(f"R²: {r2_lr:.4f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Criar o modelo Random Forest
# n_estimators=100 significa que vamos criar 100 árvores de decisão
# n_jobs=-1 usa todos os núcleos do processador para ser mais rápido
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Treinar 
rf_model.fit(X_train, y_train)

# 3. Fazer previsões
y_pred_rf = rf_model.predict(X_test)

# 4. Calcular as métricas
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("--- Random Forest Regressor ---")
print(f"MAE: {mae_rf:.2f} minutos")
print(f"RMSE: {rmse_rf:.2f} minutos")
print(f"R²: {r2_rf:.4f}")

In [ ]:
# Obter a importância das features do modelo Random Forest
importancias = rf_model.feature_importances_

# Criar um DataFrame para visualizar melhor
df_importancia = pd.DataFrame({
    'Feature': X_train.columns,
    'Importancia': importancias
})

# Ordenar da mais importante para a menos importante
df_importancia = df_importancia.sort_values(by='Importancia', ascending=False)

# Mostrar as 15 mais importantes
plt.figure(figsize=(10, 8))
sns.barplot(data=df_importancia.head(15), x='Importancia', y='Feature', palette='viridis')
plt.title('Top 15 Variáveis Mais Importantes (Random Forest)')
plt.xlabel('Importância Relativa')
plt.ylabel('Variável')
plt.show()

# Imprimir a lista
print(df_importancia.head(10))

In [ ]:
def prever_tempo_entrega(distancia, clima, transito, veiculo, preparo, hora, experiencia):
    # 1. Cria um dicionário com os dados do pedido hipotético
    pedido = {
        'Order_Hour': [hora],
        'Is_Weekend': [0], # 0 para dia de semana
        'Is_Festival': [0],
        'Rider_Experience_Years': [experiencia],
        'Rider_Rating': [4.0], # assumindo nota média
        'Restaurant_Rating': [4.0],
        'Order_Items': [1],
        'Preparation_Time_Min': [preparo],
        'Road_Distance_km': [distancia],
        'Number_of_Signals': [10], # valor médio
        # Preenchemos as categóricas com o que o usuário passou
        'Weather': [clima],
        'Traffic_Level': [transito],
        'Vehicle_Type': [veiculo],
        'Cuisine_Type': ['North Indian'], # assumindo um padrão
        'Restaurant_Load': ['Medium'],
        'Delivery_Priority': ['Normal']
    }
    
    # 2. Transforma em DataFrame
    df_novo = pd.DataFrame(pedido)
    
    # 3. Aplica o mesmo One-Hot Encoding
    # IMPORTANTE: Precisamos garantir que o novo dado tenha as MESMAS colunas que o X_train
    df_novo_encoded = pd.get_dummies(df_novo)
    df_novo_encoded = df_novo_encoded.reindex(columns=X_train.columns, fill_value=0)
    
    # 4. Faz a previsão
    previsao = rf_model.predict(df_novo_encoded)[0]
    
    return previsao

# --- TESTANDO CENÁRIOS ---

# Cenário 1: Perto, limpo, trânsito baixo, scooter
c1 = prever_tempo_entrega(distancia=2.0, clima='Clear', transito='Low', veiculo='Scooter', preparo=10, hora=14, experiencia=5)

# Cenário 2: Mesma distância, mas com chuva e trânsito moderado
c2 = prever_tempo_entrega(distancia=2.0, clima='Rain', transito='Moderate', veiculo='Scooter', preparo=10, hora=14, experiencia=5)

# Cenário 3: Distância longa (15km), limpo, trânsito baixo
c3 = prever_tempo_entrega(distancia=15.0, clima='Clear', transito='Low', veiculo='Scooter', preparo=10, hora=14, experiencia=5)

# Cenário 4: Distância longa (15km), tempestade, trânsito severo
c4 = prever_tempo_entrega(distancia=15.0, clima='Storm', transito='Severe', veiculo='Bike', preparo=10, hora=14, experiencia=5)

print(f"Cenário 1 (2km, limpo): {c1:.1f} minutos")
print(f"Cenário 2 (2km, chuva): {c2:.1f} minutos")
print(f"Cenário 3 (15km, limpo): {c3:.1f} minutos")
print(f"Cenário 4 (15km, tempestade): {c4:.1f} minutos")

In [ ]:
def prever_tempo_entrega(distancia, clima, transito, veiculo, preparo, hora, experiencia):
    # 1. Cria um dicionário com os dados do pedido hipotético
    pedido = {
        'Order_Hour': [hora],
        'Is_Weekend': [0], # 0 para dia de semana
        'Is_Festival': [0],
        'Rider_Experience_Years': [experiencia],
        'Rider_Rating': [4.0],
        'Restaurant_Rating': [4.0],
        'Order_Items': [1],
        'Preparation_Time_Min': [preparo],
        'Road_Distance_km': [distancia],
        'Number_of_Signals': [10], # valor médio
        'Weather': [clima],
        'Traffic_Level': [transito],
        'Vehicle_Type': [veiculo],
        'Cuisine_Type': ['North Indian'],
        'Restaurant_Load': ['Medium'],
        'Delivery_Priority': ['Normal']
    }
    
    # 2. Transforma em DataFrame
    df_novo = pd.DataFrame(pedido)
    
    # 3. Aplica o mesmo One-Hot Encoding
    df_novo_encoded = pd.get_dummies(df_novo)
    df_novo_encoded = df_novo_encoded.reindex(columns=X_train.columns, fill_value=0)
    
    # 4. Faz a previsão
    previsao = rf_model.predict(df_novo_encoded)[0]
    
    return previsao

# --- TESTANDO CENÁRIOS ---

# Cenário 1: Perto, limpo, trânsito baixo, scooter
c1 = prever_tempo_entrega(distancia=2.0, clima='Clear', transito='Low', veiculo='Scooter', preparo=10, hora=14, experiencia=5)

# Cenário 2: Mesma distância, mas com chuva e trânsito moderado
c2 = prever_tempo_entrega(distancia=2.0, clima='Rain', transito='Moderate', veiculo='Scooter', preparo=10, hora=14, experiencia=5)

# Cenário 3: Distância longa (15km), limpo, trânsito baixo
c3 = prever_tempo_entrega(distancia=15.0, clima='Clear', transito='Low', veiculo='Scooter', preparo=10, hora=14, experiencia=5)

# Cenário 4: Distância longa (15km), tempestade, trânsito severo
c4 = prever_tempo_entrega(distancia=15.0, clima='Storm', transito='Severe', veiculo='Bike', preparo=10, hora=14, experiencia=5)

print(f"Cenário 1 (2km, limpo): {c1:.1f} minutos")
print(f"Cenário 2 (2km, chuva): {c2:.1f} minutos")
print(f"Cenário 3 (15km, limpo): {c3:.1f} minutos")
print(f"Cenário 4 (15km, tempestade): {c4:.1f} minutos")

In [ ]:
import joblib
import os

# Criar a pasta 'models' se ela não existir
os.makedirs('../models', exist_ok=True)

# Salvar o modelo treinado
joblib.dump(rf_model, '../models/random_forest_model.pkl')

# Salvar as colunas esperadas
joblib.dump(X_train.columns.tolist(), '../models/model_columns.pkl')

print("Modelo e colunas salvos com sucesso na pasta models/!")